# Описание проекта

Заказчик этого исследования — большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ. Перед компанией стоит задача определить текущий уровень потребительской лояльности, или NPS (от англ. Net Promoter Score), среди клиентов из России. 

Чтобы определить уровень лояльности, клиентам задавали классический вопрос: «Оцените по шкале от 1 до 10 вероятность того, что вы порекомендуете компанию друзьям и знакомым».

Компания провела опрос и попросила вас подготовить дашборд с его итогами. Большую базу данных для такой задачи разворачивать не стали и выгрузили данные в SQLite. 

**Чтобы оценить результаты опроса, оценки обычно делят на три группы:**

9-10 баллов — «cторонники» (англ. promoters);

7-8 баллов — «нейтралы» (англ. passives);

0-6 баллов — «критики» (англ. detractors).

Итоговое значение NPS рассчитывается по формуле: % «сторонников» - % «критиков».
Таким образом, значение этого показателя варьируется от -100% (когда все клиенты «критики») до 100% (когда все клиенты лояльны к сервису). Но это крайние случаи, которые редко встретишь на практике. 
Интерпретируя результаты NPS-опросов, следует также помнить, что само значение мало о чём говорит. Однако исследования показывают, что клиенты-сторонники полезны любому бизнесу. Они чаще других повторно совершают покупки, активнее тестируют обновления и приводят в сервис своих друзей и знакомых. Поэтому NPS остаётся одной из важнейших метрик бизнеса. 


**Вопросы на которые необходимо ответить:**

Как распределены участники опроса по возрасту и полу? Каких пользователей больше: новых или старых? Пользователи из каких городов активнее участвовали в опросе?

Какие группы пользователей наиболее лояльны к сервису? Какие менее?

Какой общий NPS среди всех опрошенных?

Как можно описать клиентов, которые относятся к группе cторонников (англ. promoters)?

**Импортируем библиотеки:**

In [ ]:
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine, inspect

**Подключаемся к БД:**

In [ ]:
path_to_db_local = '../data/telecomm_csi.db'
path_to_db_platform = '../data/telecomm_csi.db'
path_to_db = None

if os.path.exists(path_to_db_local):
    path_to_db = path_to_db_local
elif os.path.exists(path_to_db_platform):
    path_to_db = path_to_db_platform
else:
    raise Exception('Файл с базой данных SQLite не найден!')

if path_to_db:
    engine = create_engine(f'sqlite:///{path_to_db}', echo=False)

In [ ]:
# Создаем объект инспектора базы данных
inspector = inspect(engine)

# Получаем список всех таблиц в базе данных
tables = inspector.get_table_names()

# Выводим список таблиц
print(tables)

['age_segment', 'lifetime_segment', 'location', 'traffic_segment', 'user']


**Рассмотрим все таблицы изучаемой БД:**

In [ ]:
query = """
SELECT * 
FROM user
LIMIT 10;
"""

user = pd.read_sql(query, engine)
user

  user_id  lt_day   age  gender_segment  os_name cpe_type_name  location_id  \
0  A001A2    2320  45.0             1.0  ANDROID    SMARTPHONE           55   
1  A001WF    2344  53.0             0.0  ANDROID    SMARTPHONE           21   
2  A003Q7     467  57.0             0.0  ANDROID    SMARTPHONE           28   
3  A004TB    4190  44.0             1.0      IOS    SMARTPHONE           38   
4  A004XT    1163  24.0             0.0  ANDROID    SMARTPHONE           39   
5  A005O0    5501  42.0             1.0  ANDROID    SMARTPHONE           34   
6  A0061R    1236  45.0             0.0  ANDROID    SMARTPHONE           55   
7  A009KS     313  35.0             0.0  ANDROID    SMARTPHONE           28   
8  A00AES    3238  36.0             1.0  ANDROID    SMARTPHONE           41   
9  A00F70    4479  54.0             1.0  ANDROID    SMARTPHONE            9   

   age_gr_id  tr_gr_id  lt_gr_id  nps_score  
0          5         5         8         10  
1          5         5         8      

In [ ]:
query = """
SELECT * 
FROM traffic_segment
LIMIT 10;
"""

traffic_segment = pd.read_sql(query, engine)
traffic_segment

   tr_gr_id  bucket_min  bucket_max        title
0         1        0.00        0.00         01 0
1         2        0.00        0.01    01 0-0.01
2         3        0.01        0.10  02 0.01-0.1
3         4        0.10        1.00     03 0.1-1
4         5        1.00        5.00       04 1-5
5         6        5.00       10.00      05 5-10
6         7       10.00       15.00     06 10-15
7         8       15.00       20.00     07 15-20
8         9       20.00       25.00     08 20-25
9        10       25.00       30.00     09 25-30

In [ ]:
query = """
SELECT * 
FROM location
LIMIT 10;
"""

location = pd.read_sql(query, engine)
location

   location_id         city country
0            1  Архангельск  Россия
1            2    Астрахань  Россия
2            3     Балашиха  Россия
3            4      Барнаул  Россия
4            5     Белгород  Россия
5            6       Брянск  Россия
6            7  Владивосток  Россия
7            8     Владимир  Россия
8            9    Волгоград  Россия
9           10     Волжский  Россия

In [ ]:
query = """
SELECT DISTINCT country 
FROM location
LIMIT 10;
"""

dc = pd.read_sql(query, engine)
dc

  country
0  Россия

In [ ]:
query = """
SELECT DISTINCT city 
FROM location;
"""

dcity = pd.read_sql(query, engine)
dcity

           city
0   Архангельск
1     Астрахань
2      Балашиха
3       Барнаул
4      Белгород
..          ...
57    Челябинск
58    Череповец
59         Чита
60       Якутск
61    Ярославль

[62 rows x 1 columns]

In [ ]:
for i in range(len(dcity)):
    print(dcity.iloc[i, 0])  # Вывод значений из первого (и единственного) столбца

Архангельск
Астрахань
Балашиха
Барнаул
Белгород
Брянск
Владивосток
Владимир
Волгоград
Волжский
Воронеж
Грозный
Екатеринбург
Иваново
Ижевск
Иркутск
Казань
Калининград
Калуга
Кемерово
Киров
Краснодар
Красноярск
Курск
Липецк
Магнитогорск
Махачкала
Москва
НабережныеЧелны
НижнийНовгород
НижнийТагил
Новокузнецк
Новосибирск
Омск
Оренбург
Пенза
Пермь
РостовнаДону
Рязань
Самара
СанктПетербург
Саранск
Саратов
Смоленск
Сочи
Ставрополь
Сургут
Тверь
Тольятти
Томск
Тула
Тюмень
УланУдэ
Ульяновск
Уфа
Хабаровск
Чебоксары
Челябинск
Череповец
Чита
Якутск
Ярославль


In [ ]:
query = """
SELECT * 
FROM lifetime_segment
LIMIT 10;
"""

lifetime_segment = pd.read_sql(query, engine)
lifetime_segment

   lt_gr_id  bucket_min  bucket_max     title
0         1         1.0         1.0      01 1
1         2         2.0         2.0      02 2
2         3         3.0         3.0      03 3
3         4         4.0         6.0    04 4-6
4         5         7.0        12.0   05 7-12
5         6        13.0        24.0  06 13-24
6         7        25.0        36.0  07 25-36
7         8        36.0         NaN    08 36+

In [ ]:
query = """
SELECT * 
FROM age_segment
LIMIT 10;
"""

age_segment = pd.read_sql(query, engine)
age_segment

   age_gr_id  bucket_min  bucket_max     title
0          1         NaN        15.0  01 до 16
1          2        16.0        24.0  02 16-24
2          3        25.0        34.0  03 25-34
3          4        35.0        44.0  04 35-44
4          5        45.0        54.0  05 45-54
5          6        55.0        64.0  06 55-64
6          7        66.0         NaN   07 66 +
7          8         NaN         NaN    08 n/a

In [ ]:
query = """
SELECT user_id,
       lt_day,
CASE
    WHEN lt_day >= 365 THEN 'Старый'
    WHEN lt_day <= 365 THEN 'Новый'
END AS lt_new,
       age,
CASE
    WHEN gender_segment = 0 THEN 'Мужчина'
    WHEN gender_segment = 1 THEN 'Женщина'
END AS gender_segment,
       os_name,
       cpe_type_name,
       l.country,
       l.city,
       a_s.title AS age_segment,
       t_s.title AS traffic_segment,
       lt_s.title AS lifetime_segment,
       nps_score,
       CASE
            WHEN nps_score >= 9 THEN 'Сторонники'
            WHEN nps_score > 6 AND nps_score < 9 THEN 'Нейтралы'
            WHEN nps_score <= 6 THEN 'Критики'
       END AS nps_group
FROM user AS u JOIN (
        SELECT location_id, 
        CASE
            WHEN city = 'НабережныеЧелны' THEN 'Набережные Челны'
            WHEN city = 'НижнийНовгород' THEN 'Нижний Новгород'
            WHEN city = 'НижнийТагил' THEN 'Нижний Тагил'
            WHEN city = 'РостовнаДону' THEN 'Ростов-на-Дону'
            WHEN city = 'СанктПетербург' THEN 'Санкт-Петербург'
            WHEN city = 'УланУдэ' THEN 'Улан-Удэ'
            ELSE city END AS city,
        country
        FROM location
    ) AS l ON u.location_id = l.location_id 
    JOIN age_segment AS a_s ON u.age_gr_id = a_s.age_gr_id
    JOIN traffic_segment AS t_s ON u.tr_gr_id = t_s.tr_gr_id
    JOIN lifetime_segment AS lt_s ON u.lt_gr_id = lt_s.lt_gr_id
WHERE gender_segment IS NOT NULL AND age IS NOT NULL;
"""
df = pd.read_sql(query, engine)
df.head(20)

   user_id  lt_day  lt_new   age gender_segment  os_name cpe_type_name  \
0   A001A2    2320  Старый  45.0        Женщина  ANDROID    SMARTPHONE   
1   A001WF    2344  Старый  53.0        Мужчина  ANDROID    SMARTPHONE   
2   A003Q7     467  Старый  57.0        Мужчина  ANDROID    SMARTPHONE   
3   A004TB    4190  Старый  44.0        Женщина      IOS    SMARTPHONE   
4   A004XT    1163  Старый  24.0        Мужчина  ANDROID    SMARTPHONE   
5   A005O0    5501  Старый  42.0        Женщина  ANDROID    SMARTPHONE   
6   A0061R    1236  Старый  45.0        Мужчина  ANDROID    SMARTPHONE   
7   A009KS     313   Новый  35.0        Мужчина  ANDROID    SMARTPHONE   
8   A00AES    3238  Старый  36.0        Женщина  ANDROID    SMARTPHONE   
9   A00F70    4479  Старый  54.0        Женщина  ANDROID    SMARTPHONE   
10  A00HL5    5297  Старый  39.0        Мужчина  ANDROID    SMARTPHONE   
11  A00NDN    1374  Старый  21.0        Мужчина  ANDROID    SMARTPHONE   
12  A00P46     179   Новый  27.0      

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501152 entries, 0 to 501151
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           501152 non-null  object 
 1   lt_day            501152 non-null  int64  
 2   lt_new            501152 non-null  object 
 3   age               501152 non-null  float64
 4   gender_segment    501152 non-null  object 
 5   os_name           501152 non-null  object 
 6   cpe_type_name     501152 non-null  object 
 7   country           501152 non-null  object 
 8   city              501152 non-null  object 
 9   age_segment       501152 non-null  object 
 10  traffic_segment   501152 non-null  object 
 11  lifetime_segment  501152 non-null  object 
 12  nps_score         501152 non-null  int64  
 13  nps_group         501152 non-null  object 
dtypes: float64(1), int64(2), object(11)
memory usage: 53.5+ MB


In [ ]:
df.to_csv('../data/project_table.csv', index=False)

**Презентация:** ссылка-на-дашборд-удалена-при-обезличивании